# Human-in-the-loop correction notebook

**Goal**: review a real-plan inference, mark false positives as DELETE and add missed columns as ADD, and write the corrections to `data/corrections.db`. Run `scripts/retrain_yolo.py` afterwards to fold the corrections into a fine-tune.

Closed feedback loop:
```
review (this notebook) → data/corrections.db → scripts/retrain_yolo.py
  → column_detect_ft_{ts}.pt → manual `cp` to column_detect.pt → next review
```

Schema contract (kept compatible with `scripts/retrain_yolo.py`):
```
data/corrections.db                     — SQLite log of edits/deletes/adds
data/jobs/{job_id}/render.jpg           — the plan image as reviewed
data/jobs/{job_id}/px_detections.json   — { "columns": [{"bbox": [...], ...}, ...] }
```

In [ ]:
# Cell 1 — imports
import io
import math
import sys
from pathlib import Path

import numpy as np
import torch
import torchvision.ops as tvops
from PIL import Image, ImageDraw
from ultralytics import YOLO
import ipywidgets as widgets
from IPython.display import display, clear_output

Image.MAX_IMAGE_PIXELS = None

# Make scripts/ importable
_here = Path.cwd()
if str(_here / 'scripts') not in sys.path:
    sys.path.insert(0, str(_here / 'scripts'))
from corrections_logger import (
    new_job_id, save_job, record_delete, record_add, record_edit, summary,
)

In [ ]:
# Cell 2 — config
WEIGHTS    = Path('column_detect.pt')
IMAGE_PATH = Path('/home/jiezhi/Documents/TGCH floor plan/L3.jpg')

TILE_SIZE  = 1280
TILE_STEP  = 1080            # 200 px overlap, same as training
CONF_TH    = 0.25
IOU_TH     = 0.45
DEVICE     = 0 if torch.cuda.is_available() else 'cpu'

print(f'weights : {WEIGHTS}')
print(f'image   : {IMAGE_PATH}')
print(f'device  : {DEVICE}')

In [ ]:
# Cell 3 — load + tiled inference (collects raw all_boxes/all_scores)
model = YOLO(str(WEIGHTS))
img   = Image.open(IMAGE_PATH).convert('RGB')
W, H  = img.size
print(f'image size: {W} × {H} px')

xs = list(range(0, W - TILE_SIZE, TILE_STEP))
if not xs or xs[-1] + TILE_SIZE < W:
    xs.append(max(0, W - TILE_SIZE))
ys = list(range(0, H - TILE_SIZE, TILE_STEP))
if not ys or ys[-1] + TILE_SIZE < H:
    ys.append(max(0, H - TILE_SIZE))
print(f'tiles   : {len(xs)} × {len(ys)} = {len(xs) * len(ys)}')

all_boxes, all_scores = [], []
for ty in ys:
    for tx in xs:
        tile = img.crop((tx, ty, tx + TILE_SIZE, ty + TILE_SIZE))
        result = model.predict(
            source=tile, imgsz=TILE_SIZE, conf=CONF_TH, iou=IOU_TH,
            device=DEVICE, verbose=False,
        )[0]
        if result.boxes is None or len(result.boxes) == 0:
            continue
        xyxy = result.boxes.xyxy.cpu().numpy()
        conf = result.boxes.conf.cpu().numpy()
        xyxy[:, [0, 2]] += tx
        xyxy[:, [1, 3]] += ty
        all_boxes.extend(xyxy.tolist())
        all_scores.extend(conf.tolist())
print(f'raw detections (pre-NMS): {len(all_boxes)}')

## Cell 4 — 7-filter post-processing

Duplicates `test_column.ipynb` cell 5 intentionally for v1. Future refactor: lift into `scripts/postprocess_pipeline.py` so both notebooks import the same code. See plan notes.

In [ ]:
# Cell 4 — post-processing (mirrors test_column.ipynb cell 5)
USE_STAIR_MASK   = False
USE_OCR_FILTER   = True

GREY_DARK_THRESH = 200
MIN_FILL_RATIO   = 0.40
MIN_BORDER_RATIO = 0.35
BORDER_THICK_PX  = 2
MAX_ASPECT       = 2.0
MIN_SIDE_PX      = 12
MAX_SIDE_PX      = 60
CENTRE_DIST_PX   = 50
NMS_IOU_BACKUP   = 0.15

OCR_CROP_PAD_PX  = 4
OCR_MIN_CONF     = 50
OCR_MIN_CHARS    = 2
OCR_CHAR_WHITELIST = (
    'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    'abcdefghijklmnopqrstuvwxyz0123456789-'
)

img_gray = np.asarray(img.convert('L'))

def _bbox_aspect(b):
    w = max(1.0, b[2] - b[0]); h = max(1.0, b[3] - b[1])
    return max(w, h) / min(w, h)

def _bbox_side_range(b):
    w = b[2] - b[0]; h = b[3] - b[1]
    return min(w, h), max(w, h)

def _shape_passes(b):
    ix1, iy1 = max(0, int(b[0])), max(0, int(b[1]))
    ix2, iy2 = min(img_gray.shape[1], int(b[2])), min(img_gray.shape[0], int(b[3]))
    if ix2 - ix1 < 4 or iy2 - iy1 < 4:
        return False
    patch = img_gray[iy1:iy2, ix1:ix2]
    dark = (patch <= GREY_DARK_THRESH)
    fill_ratio = float(dark.sum()) / dark.size
    if fill_ratio >= MIN_FILL_RATIO:
        return True
    h, w = patch.shape
    bt = min(BORDER_THICK_PX, min(h, w) // 2)
    border_mask = np.zeros_like(dark)
    border_mask[:bt, :] = True; border_mask[-bt:, :] = True
    border_mask[:, :bt] = True; border_mask[:, -bt:] = True
    border_total = border_mask.sum()
    border_dark = (dark & border_mask).sum()
    return float(border_dark) / max(1, border_total) >= MIN_BORDER_RATIO

try:
    import pytesseract
    _OCR_OK = True
except ImportError:
    _OCR_OK = False
    print('  [OCR filter skipped: pytesseract not installed]')

_OCR_CONFIG = (
    f'--oem 3 --psm 6 -c tessedit_char_whitelist={OCR_CHAR_WHITELIST}'
)

def _bbox_has_text(b):
    if not _OCR_OK:
        return False
    pad = OCR_CROP_PAD_PX
    ix1 = max(0, int(b[0]) - pad); iy1 = max(0, int(b[1]) - pad)
    ix2 = min(img_gray.shape[1], int(b[2]) + pad)
    iy2 = min(img_gray.shape[0], int(b[3]) + pad)
    if ix2 - ix1 < 6 or iy2 - iy1 < 6:
        return False
    crop = img_gray[iy1:iy2, ix1:ix2]
    try:
        data = pytesseract.image_to_data(
            crop, config=_OCR_CONFIG, output_type=pytesseract.Output.DICT,
        )
    except Exception:
        return False
    chars = 0
    for txt, conf in zip(data.get('text', []), data.get('conf', [])):
        if not txt:
            continue
        try:
            c = float(conf)
        except (TypeError, ValueError):
            continue
        if c < OCR_MIN_CONF:
            continue
        chars += sum(1 for ch in txt if ch.isalnum())
        if chars >= OCR_MIN_CHARS:
            return True
    return False

def _centre_dist_nms(boxes, scores, max_d):
    order = sorted(range(len(boxes)), key=lambda i: -scores[i])
    kept_idx, kept_c = [], []
    for i in order:
        b = boxes[i]
        cx, cy = (b[0]+b[2])/2.0, (b[1]+b[3])/2.0
        if not any(math.hypot(cx-kx, cy-ky) <= max_d for kx, ky in kept_c):
            kept_idx.append(i)
            kept_c.append((cx, cy))
    return kept_idx

if not all_boxes:
    boxes_final  = np.zeros((0, 4), dtype=np.float32)
    scores_final = np.zeros((0,),    dtype=np.float32)
    print('raw detections : 0  — pipeline skipped')
else:
    boxes_arr  = np.array(all_boxes,  dtype=np.float32)
    scores_arr = np.array(all_scores, dtype=np.float32)
    print(f'raw                   : {len(boxes_arr)}')

    mask = np.array([_bbox_aspect(b) <= MAX_ASPECT for b in boxes_arr])
    boxes_arr = boxes_arr[mask]; scores_arr = scores_arr[mask]
    print(f'after aspect          : {len(boxes_arr)}')

    def _sz_ok(b):
        mn, mx = _bbox_side_range(b)
        return MIN_SIDE_PX <= mn and mx <= MAX_SIDE_PX
    mask = np.array([_sz_ok(b) for b in boxes_arr]) if len(boxes_arr) else np.zeros(0, dtype=bool)
    boxes_arr = boxes_arr[mask]; scores_arr = scores_arr[mask]
    print(f'after size            : {len(boxes_arr)}')

    mask = np.array([_shape_passes(b) for b in boxes_arr]) if len(boxes_arr) else np.zeros(0, dtype=bool)
    boxes_arr = boxes_arr[mask]; scores_arr = scores_arr[mask]
    print(f'after shape           : {len(boxes_arr)}')

    if USE_OCR_FILTER and _OCR_OK and len(boxes_arr):
        text_mask = np.array([_bbox_has_text(b) for b in boxes_arr])
        boxes_arr  = boxes_arr [~text_mask]
        scores_arr = scores_arr[~text_mask]
        print(f'after OCR text       : {len(boxes_arr)}  (dropped {int(text_mask.sum())} text bboxes)')

    idx = _centre_dist_nms(boxes_arr.tolist(), scores_arr.tolist(), CENTRE_DIST_PX)
    boxes_arr = boxes_arr[idx]; scores_arr = scores_arr[idx]
    print(f'after centre-NMS      : {len(boxes_arr)}')

    if len(boxes_arr) == 0:
        boxes_final  = np.zeros((0, 4), dtype=np.float32)
        scores_final = np.zeros((0,),    dtype=np.float32)
    else:
        boxes_t  = torch.tensor(boxes_arr,  dtype=torch.float32)
        scores_t = torch.tensor(scores_arr, dtype=torch.float32)
        keep = tvops.nms(boxes_t, scores_t, iou_threshold=NMS_IOU_BACKUP)
        boxes_final  = boxes_t[keep].numpy()
        scores_final = scores_t[keep].numpy()
    print(f'FINAL columns         : {len(boxes_final)}')

In [ ]:
# Cell 5 — register the job (writes data/jobs/{job_id}/render.jpg + px_detections.json)
job_id = new_job_id()
job_dir = save_job(job_id, img, boxes_final, scores_final,
                   source_path=str(IMAGE_PATH))
print(f'job_id   : {job_id}')
print(f'job dir  : {job_dir}')
print(f'detections registered: {len(boxes_final)}')

## Cell 6 — review grid

Page through detections in batches of 20. Set the dropdown under each thumbnail to **delete** if the bbox is a false positive (text, wall corner, geometry artefact, etc.). Leave **keep** for real columns. The red square in each crop is the detected bbox; the surrounding context shows what's actually under it.

Click **Save corrections** when done. Per-click writes were rejected on purpose (lets you change your mind, no DB spam).

In [ ]:
# Cell 6 — build the review grid (ipywidgets)
PAGE_SIZE  = 20
GRID_COLS  = 5
CROP_PAD   = 14
THUMB_SIZE = 96

# Pre-render thumbnails once (fast — ~500 crops at 96×96 PNG ~5 MB total).
def _make_crop_bytes(b):
    x1, y1, x2, y2 = b
    cx1 = max(0, int(x1 - CROP_PAD)); cy1 = max(0, int(y1 - CROP_PAD))
    cx2 = min(W, int(x2 + CROP_PAD)); cy2 = min(H, int(y2 + CROP_PAD))
    crop = img.crop((cx1, cy1, cx2, cy2)).copy()
    d = ImageDraw.Draw(crop)
    d.rectangle([(x1 - cx1, y1 - cy1), (x2 - cx1, y2 - cy1)],
                outline=(220, 30, 30), width=2)
    crop = crop.resize((THUMB_SIZE, THUMB_SIZE))
    buf = io.BytesIO()
    crop.save(buf, format='PNG')
    return buf.getvalue()

_thumb_cache = [_make_crop_bytes(b) for b in boxes_final]
marks = ['keep'] * len(boxes_final)
page_state = {'idx': 0}

grid_out = widgets.Output()
status   = widgets.Label('')

def _make_tile(i):
    img_w = widgets.Image(value=_thumb_cache[i], format='png',
                           width=THUMB_SIZE, height=THUMB_SIZE)
    dd = widgets.Dropdown(
        options=[('keep', 'keep'), ('DELETE', 'delete')],
        value=marks[i],
        layout=widgets.Layout(width=f'{THUMB_SIZE}px'),
    )
    def _on_change(change, idx=i):
        marks[idx] = change['new']
    dd.observe(_on_change, names='value')
    label = widgets.Label(f'#{i}  c={float(scores_final[i]):.2f}',
                          layout=widgets.Layout(width=f'{THUMB_SIZE}px'))
    return widgets.VBox([img_w, dd, label],
                        layout=widgets.Layout(margin='4px',
                                               border='1px solid #ddd',
                                               padding='4px'))

def _render():
    n = len(boxes_final)
    pages = max(1, (n + PAGE_SIZE - 1) // PAGE_SIZE)
    start = page_state['idx'] * PAGE_SIZE
    end   = min(start + PAGE_SIZE, n)
    tiles = [_make_tile(i) for i in range(start, end)]
    rows = [widgets.HBox(tiles[r:r+GRID_COLS])
            for r in range(0, len(tiles), GRID_COLS)]
    with grid_out:
        clear_output(wait=True)
        display(widgets.VBox(rows))
        n_del = sum(1 for m in marks if m == 'delete')
        print(f'Page {page_state["idx"]+1}/{pages}    '
              f'showing {start}-{end-1} of {n}    '
              f'marked DELETE so far: {n_del}')

def _go_prev(_):
    if page_state['idx'] > 0:
        page_state['idx'] -= 1; _render()
def _go_next(_):
    n = len(boxes_final)
    pages = max(1, (n + PAGE_SIZE - 1) // PAGE_SIZE)
    if page_state['idx'] < pages - 1:
        page_state['idx'] += 1; _render()
def _save(_):
    n_deleted = 0
    for i, m in enumerate(marks):
        if m == 'delete':
            try:
                record_delete(job_id, i)
                n_deleted += 1
            except Exception as e:
                print(f'  ! delete {i} failed: {e}')
    s = summary()
    status.value = (f'Saved {n_deleted} delete corrections.  '
                    f'DB now has {s["deletes"]} deletes / '
                    f'{s["edits_or_adds"]} edits-or-adds across '
                    f'{s["jobs"]} job(s).')
    print(status.value)

prev_btn = widgets.Button(description='← Prev')
next_btn = widgets.Button(description='Next →')
save_btn = widgets.Button(description='Save corrections', button_style='primary')
prev_btn.on_click(_go_prev)
next_btn.on_click(_go_next)
save_btn.on_click(_save)

display(widgets.HBox([prev_btn, next_btn, save_btn, status]))
display(grid_out)
_render()

## Cell 7 — ADD missed columns (FN corrections)

For columns the model **missed entirely** (no thumbnail above to mark), open `L3.jpg` in your image viewer, hover each missed column, read off (cx, cy) and a rough size_px (e.g., 24 for a small filled square, 32 for a typical outlined one). Add tuples to the `missed` list below, then run the cell. Each entry becomes a row in `data/corrections.db` with `is_delete=0` and a `bbox` derived from (cx, cy, size_px).

You can iterate: run this cell, check `summary()`, add more entries, run again. Each call appends; nothing is overwritten.

In [ ]:
# Cell 7 — ADD missed columns
# Format: (cx, cy, size_px). The bbox becomes (cx - sz/2, cy - sz/2, cx + sz/2, cy + sz/2).
missed = [
    # (1234, 5678, 28),
]

n_added = 0
for cx, cy, sz in missed:
    bbox = [cx - sz/2.0, cy - sz/2.0, cx + sz/2.0, cy + sz/2.0]
    record_add(job_id, bbox)
    n_added += 1
print(f'Added {n_added} missed-column corrections.')
print('Corrections DB summary:', summary())

## Cell 8 — retrain handoff

When `scripts/retrain_yolo.py` sees enough corrections (default ≥10, configurable via `--min-corrections`), it folds them into a YOLO fine-tune from `column_detect.pt` and writes a timestamped weight to the project root. Promote it manually after inspecting on a real plan.

In [ ]:
# Cell 8 — print the retrain command
s = summary()
print(f'Current DB: {s["jobs"]} job(s), {s["corrections"]} corrections '
      f'({s["deletes"]} deletes, {s["edits_or_adds"]} edits/adds)')
print()
print('To fold corrections into a fine-tune, run from project root:')
print('  python3 scripts/retrain_yolo.py --epochs 30 --min-corrections 10')
print()
print('Dry-run (build dataset, skip training):')
print('  python3 scripts/retrain_yolo.py --dry-run')
print()
print('Output weight will be column_detect_ft_{timestamp}.pt at project root.')
print('Inspect on a real plan, then manually promote:')
print('  cp column_detect_ft_<ts>.pt column_detect.pt')